# Seminar HCI and BCI in practice
## Session 6 Classification

***In this session the antagonistic finger movements are finally discriminated by means of three different classification algorithms.***


In [ ]:
import numpy as np
import os
import sys
import pickle
from scipy import stats

sys.path.append(os.path.join(os.getcwd(), "src"))
from nearly import nearly
from getBalancedTrainset import getBalancedTrainset
from train_bayes import train_bayes
from test_bayes import test_bayes
from train_lda import train_lda
from test_lda import test_lda
from classification_svm import classification_svm

main_path = os.getcwd()
data_path = os.path.join(main_path, 'data')
print(f'Now you are located: {main_path}')


In [ ]:
ecog_file = os.path.join(data_path, 'raw/ecogStruct3.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

# Take a look into our data again
print("ecog contains")
for key, value in ecog.items():
    print(f"Key:{key}, Type:{type(value)}")

# Load epoch info
epoch_file = os.path.join(data_path, 'raw/epoch2.pkl')
with open(epoch_file, 'rb') as f:
    epoch = pickle.load(f)

print("\nepoch info:")
for key, value in epoch.items():
    print(f"Key:{key}, Type:{type(value)}")

In [ ]:
# Number of trials with finger movement
nTrials = np.array(ecog['periodogram']['periodogram']).shape[2]

# Frequency features (40-160 Hz based on Session 5 results)
freqBand = np.arange(40, 161)  # 40-160 Hz (inclusive)

# Find nearest frequency indices
freqIdx = np.unique(nearly(freqBand, ecog['periodogram']['centerFrequency']))

# Alternative:
# freqIdx = np.unique([np.argmin(np.abs(ecog['periodogram']['centerFrequency'] - f)) 
#                          for f in freqBand])
nFreq = len(freqIdx)

# Channel features (based on Session 5 results)
chan = np.array([17, 23, 29, 30, 39]) - 1  # Convert to 0-based indexing
nChan = len(chan)

# Prepare data for z-scoring (same as Session 4)
# Reshape to (nFreq, nChan*nTrials)
dat = np.array(ecog['periodogram']['periodogram'])[freqIdx, :, :][:, chan, :]
dat = dat.reshape(nFreq, nChan * nTrials, order='F')

# Z-score data along frequency axis
dat = stats.zscore(dat, axis=1) 

# Reshape data back to original structure with permutations
dat = dat.reshape(nFreq, nChan, nTrials, order='F')
dat = np.transpose(dat, (2, 1, 0)) 
dat = dat.reshape(nTrials, nFreq * nChan, order='F')

In [ ]:
## Create subsets for cross-validation
realClassLabels = np.array(epoch['label']) # Class labels
N = 10       # CV steps

selector = np.ceil((np.arange(1, len(realClassLabels)+1)) / (len(realClassLabels)/N))
selector = selector.astype(int)
selector = selector[np.random.permutation(len(realClassLabels))]

<h2 style="color: #FF0000; font-weight: bold;">TASK 1 (2 pt):</h2>

- What does the variable selector contain? 
- What is the purpose of cross-validation? Describe this method and its advantages/disadvantages.

In [ ]:
# --- TASK 1: what is inside selector? ---
print("selector length :", len(selector), " (one number per trial)")
print("different values:", np.unique(selector))
print("first 20 values :", selector[:20])

values, counts = np.unique(selector, return_counts=True)
print("\nhow many trials in each group:", dict(zip(values.tolist(), counts.tolist())))

# what happens in one CV step
testIdx = np.where(selector == 1)[0]
trainIdx = np.setdiff1d(np.arange(len(realClassLabels)), testIdx)
print("\nin step 1: %d trials are used for testing and %d for training" % (len(testIdx), len(trainIdx)))

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

**What does the variable `selector` contain?**

`selector` has one number for every trial, so it has **314 values**. Each value is a number **between 1 and 10**, and it says which group that trial belongs to. `N = 10`, so the trials are put into 10 groups of about the same size, around **31 or 32 trials each**.

The line `np.ceil(np.arange(1, len(realClassLabels)+1) / (len(realClassLabels)/N))` first gives 1 1 1 ... 2 2 2 ... up to 10, so the trials would be in order. Then `selector[np.random.permutation(len(realClassLabels))]` **shuffles** them, so the groups are mixed and not just the first 31 trials, the next 31 and so on. This is important because the trials were recorded one after the other, and without shuffling one group could contain only trials from the beginning of the recording.

In the loop below, `selector` is used to split the data: in step `cv`, all trials with `selector == cv` are the **test set** and all the others are the **training set**. So in step 1 there are 31 test trials and 283 training trials.

**What is the purpose of cross-validation?**

The problem is that we cannot test the classifier on the same data we trained it on. If we did, the classifier could just remember the trials and the accuracy would look very good, but it would not tell us anything about new data. So we need data the classifier has never seen.

We only have 314 trials, which is not a lot. If we just kept 30 trials aside for testing, we would lose them for training, and the result would also depend a lot on which 30 trials we happened to pick.

**Cross-validation** solves this. The method works like this:

1. Split all the trials into 10 groups (that is what `selector` does).
2. Take group 1 as the test set and the other 9 groups as the training set. Train the classifier and predict the labels of group 1.
3. Repeat with group 2 as the test set, then group 3, and so on until group 10.
4. In the end every trial has been predicted exactly once, and always by a classifier that did not see it during training. Then the accuracy is calculated over all trials together.

**Advantages**

- Every trial is used for testing once and for training 9 times, so no data is wasted. This is very useful when there is not much data.
- The result is more reliable, because it is an average over 10 different splits and does not depend on one lucky or unlucky split.
- It shows how well the classifier works on **new** data, which is what we actually want to know.

**Disadvantages**

- It takes 10 times longer, because the classifier has to be trained 10 times instead of once.
- The 10 results are not fully independent, because the training sets overlap a lot (they share most of the trials).
- The split is random, so if you run it again with a different shuffle you get a slightly different accuracy.
- Here the trials come from one continuous recording, so trials that are close in time can be similar. If such trials end up in the training and the test set, the accuracy can look a bit better than it really is.

---

## Classification lda & bayes

In [ ]:
alg = 'bayes'     # 'bayes', 'lda'
banlance = True
predictedClassLabels = np.zeros(len(realClassLabels))

for cv in range(1, N+1):
    print(f'CV step #{cv}')
    testIdx = np.where(selector == cv)[0]
    trainIdx = np.setdiff1d(np.arange(len(realClassLabels)), testIdx)

    if banlance:
        labelIdx = getBalancedTrainset(realClassLabels[trainIdx])
        trainIdx = trainIdx[labelIdx]

    # Train Classifier
    curTrain = dat[trainIdx, :]
    curClassLabels = realClassLabels[trainIdx]
    
    if alg.lower() == 'bayes':
        R = train_bayes(curTrain, curClassLabels)
    elif alg.lower() == 'lda':
        R = train_lda(curTrain, curClassLabels)

    # Testing
    curTest = dat[testIdx, :]
    
    if alg.lower() == 'bayes':
        Res = test_bayes(R, curTest)
    elif alg.lower() == 'lda':
        Res = test_lda(R, curTest)
        
    predictedClassLabels[testIdx] = Res['prediction']

# accuracy rate
accuracy = np.sum(predictedClassLabels == realClassLabels) / len(realClassLabels)
print(f'{alg} Classification accuracy: {accuracy:.2%}')

<h2 style="color: #FF0000; font-weight: bold;">TASK 2 (2 pt):</h2>

- What does it mean to balance datasets?
- Why is this sometimes done?
- Have a look at the `train_lda` and `train_bayes` functions. What kind of information do they store in the `dict` `R`?
- What is the difference between them?

In [ ]:
# --- TASK 2: what does balancing do, and what is stored in R? ---
# take the training set of the first CV step
testIdx = np.where(selector == 1)[0]
trainIdx = np.setdiff1d(np.arange(len(realClassLabels)), testIdx)

before = realClassLabels[trainIdx]
values, counts = np.unique(before, return_counts=True)
print("training labels BEFORE balancing:", dict(zip(values.tolist(), counts.tolist())),
      " total", len(before))

labelIdx = getBalancedTrainset(before.copy())
after = realClassLabels[trainIdx[labelIdx]]
values, counts = np.unique(after, return_counts=True)
print("training labels AFTER  balancing:", dict(zip(values.tolist(), counts.tolist())),
      " total", len(after))

# what the two training functions give back
balancedIdx = trainIdx[labelIdx]
R_bayes = train_bayes(dat[balancedIdx, :], realClassLabels[balancedIdx])
print("\ntrain_bayes returns a list with %d entries, one per class:" % len(R_bayes))
for r in R_bayes:
    print("   label %d -> meanC %s, varC %s" % (r.label, r.meanC.shape, r.varC.shape))

R_lda = train_lda(dat[balancedIdx, :], realClassLabels[balancedIdx])
print("\ntrain_lda returns a dict with the keys:", list(R_lda.keys()))
print("   a has shape %s, group1 and group2 are single numbers" % (R_lda['a'].shape,))

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

**What does it mean to balance datasets?**

Balancing means making the classes the **same size**. Our two classes are not equally big: in the whole data there are 163 flexion trials (label 20) and 151 extension trials (label 21). In the training set of the first CV step there are **148 flexion and 135 extension** trials.

`getBalancedTrainset` looks how many trials the **smallest** class has and then throws away trials from the bigger class until both have the same number. Here it removes 13 flexion trials, so the training set goes from 283 trials to **270 trials, 135 in each class**. By default it removes them randomly (`method='rand'`), but it can also remove them from the beginning, from the end, or evenly spread out.

It is only used for **training**. The test set is not balanced, because the test set should stay the way the real data is.

**Why is this sometimes done?**

Because a classifier can get lazy if one class is bigger. If 90% of the trials were flexion, the classifier could simply say "flexion" every time and would already be right 90% of the time, without learning anything useful. The accuracy would look great but the classifier would be useless.

Balancing stops this. When both classes are the same size, the classifier cannot get an advantage just by preferring the bigger class, and it has to actually use the features to tell them apart. It also makes the accuracy easier to read: with two equal classes, guessing gives 50%, so anything clearly above 50% means the classifier really learned something.

In our case the difference is small (148 against 135), so the effect is not dramatic here, but it is the correct thing to do.

**What information do `train_lda` and `train_bayes` store in `R`?**

**`train_bayes`** returns a **list with one entry per class** (so 2 entries here). For each class it stores:

- `label` - which class it is (20 or 21),
- `meanC` - the mean of **every feature** for that class, shape (310,),
- `varC` - the variance of every feature for that class, also shape (310,).

So it describes each class separately by "what is the average value of each feature and how much does it vary". Very small variances are replaced by a tiny number (`EPSILON = 5e-13`) so that later there is no division by zero.

**`train_lda`** returns a **dict with 3 keys**:

- `a` - the discriminant weights, shape (310, 1). This is one weight for every feature, and it is the direction that separates the two classes best.
- `group1` and `group2` - the projections of the two class means onto that direction, so two single numbers. They are used later as the reference points to decide which class a new trial is closer to.

**What is the difference between them?**

They look at the problem in a different way:

- **Bayes** describes **each class on its own**. It keeps a mean and a variance per feature per class and asks "which class fits this trial better". It is a *naive* Bayes, which means it treats every feature as independent and ignores that the features could be correlated with each other. That is not really true for our data (neighbouring frequencies are similar), but it still often works.
- **LDA** does not describe the classes separately, it looks for **one direction** in the feature space where the two classes are as far apart as possible. It uses the within-class scatter matrix `s_w`, so it *does* take the correlations between the features into account. In the end it squeezes all 310 features into a single number per trial and decides from that.

So Bayes stores statistics per class and per feature, while LDA stores one weight vector plus two reference values. Bayes can also handle more than two classes, but `train_lda` explicitly raises an error if there are not exactly 2 classes.

<h2 style="color: #FF0000; font-weight: bold;">TASK 3 (1 pt):</h2>

Have a look at the `test_lda` and `test_bayes` functions. What information is stored in `Res['prediction']`?

In [ ]:
# --- TASK 3: what is inside Res['prediction']? ---
# use the first CV step again
testIdx = np.where(selector == 1)[0]
trainIdx = np.setdiff1d(np.arange(len(realClassLabels)), testIdx)
labelIdx = getBalancedTrainset(realClassLabels[trainIdx].copy())
balancedIdx = trainIdx[labelIdx]

R_bayes = train_bayes(dat[balancedIdx, :], realClassLabels[balancedIdx])
Res_bayes = test_bayes(R_bayes, dat[testIdx, :])

R_lda = train_lda(dat[balancedIdx, :], realClassLabels[balancedIdx])
Res_lda = test_lda(R_lda, dat[testIdx, :])

print("number of test trials in this step:", len(testIdx))
print("\nRes keys (bayes):", list(Res_bayes.keys()))
print("prediction shape :", Res_bayes['prediction'].shape)
print("which values     :", np.unique(Res_bayes['prediction']))
print("\nfirst 10 predictions bayes:", Res_bayes['prediction'][:10])
print("first 10 predictions lda  :", Res_lda['prediction'][:10])
print("first 10 true labels      :", realClassLabels[testIdx][:10])

print("\ncorrect in this step, bayes: %d of %d"
      % ((Res_bayes['prediction'] == realClassLabels[testIdx]).sum(), len(testIdx)))
print("correct in this step, lda  : %d of %d"
      % ((Res_lda['prediction'] == realClassLabels[testIdx]).sum(), len(testIdx)))

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

**What information is stored in `Res['prediction']`?**

`Res` is a dict with only one key, `prediction`. It contains the **class label that the classifier guessed for every test trial**, so one number per trial. In our data the values are only **20 (flexion) or 21 (extension)**, and in the first CV step there are 31 test trials, so `Res['prediction']` has the shape (31,).

It is important that these are the **guessed** labels, not the true ones. The true labels are in `realClassLabels`. In the loop the predictions are written into the right places with

```python
predictedClassLabels[testIdx] = Res['prediction']
```

so after all 10 CV steps every trial has a predicted label, and the accuracy is calculated by comparing `predictedClassLabels` with `realClassLabels`.

**How the two functions get to that prediction:**

- **`test_bayes`**: for every feature and every class it computes how likely the measured value is, using the mean and variance that were stored during training. It multiplies these over all 310 features, normalises them into probabilities, and then picks for each trial the class with the **highest probability**. The labels come from `R[c].label`, so they are whatever labels were used in the training.
- **`test_lda`**: it projects the test trial onto the direction `a` that was found during training, so each trial becomes a single number. Then it compares how far this number is from `group1` and from `group2` and takes the **closer** one.

One thing I noticed: in `test_lda` the labels **20 and 21 are written directly into the code** (`resTest[...] = 20` and `= 21`), while `test_bayes` takes the labels from the trained model. So `test_lda` would give wrong labels if the classes had different numbers, and it only works for this flexion/extension problem.

In my run Bayes got 22 of 31 trials right in this step and LDA only 16 of 31. The LDA result is bad because we use 310 features but only 270 training trials, so the within-class scatter matrix `s_w` cannot be inverted properly, which also shows up in the very large values of `group1` and `group2`.

---

## Classification SVM

In [ ]:
svm_results = classification_svm(dat, realClassLabels, selector, N, optimizeC=False)
predictedClassLabels = svm_results['predictedClassLabels']
accuracy_svm = svm_results['accuracy']
print(f"Final Accuracy: {accuracy_svm:.2%}")
predictedClassLabels.shape

In [ ]:
# Take a look into `svm_results`, try to understand the output from function `classification_svm`
for key, value in svm_results.items():
    print(f"Key:{key}, Type:{type(value)}")

<h2 style="color: #FF0000; font-weight: bold;">TASK 4 (1 pt):</h2>

What is the cost parameter C and why is it iteratively optimized? Does it improve predictions you make?

In [ ]:
# --- TASK 4: does optimizing C really help? ---
from sklearn.svm import SVC
from ecogGetDefC import ecogGetDefC

# 1) run the SVM once with the default C and once with the optimisation
res_default = classification_svm(dat, realClassLabels, selector, N, optimizeC=False)
res_opt     = classification_svm(dat, realClassLabels, selector, N, optimizeC=True)

print("\naccuracy with default C   : %.2f%%" % (res_default['accuracy'] * 100))
print("accuracy with 'optimised' C: %.2f%%" % (res_opt['accuracy'] * 100))
print("best C chosen in each fold :", np.round(res_opt['best_Cs'], 4))
print("-> the chosen C jumps around a lot between the folds")

# 2) do it honestly: choose C only on the training data (nested cross-validation)
predHonest = np.zeros(len(realClassLabels), dtype=int)
for k in range(1, N + 1):
    testIdx = np.where(selector == k)[0]
    trainIdx = np.setdiff1d(np.arange(len(realClassLabels)), testIdx)
    X_train, y_train = dat[trainIdx], realClassLabels[trainIdx]

    defC = ecogGetDefC(X_train)
    grid = defC * np.array([1/27, 1/9, 1/3, 1, 3, 9, 27])

    # split the TRAINING data again into 3 inner parts
    inner = np.array_split(np.random.permutation(len(trainIdx)), 3)
    bestC, bestAcc = defC, -1
    for C in grid:
        accs = []
        for j in range(3):
            val = inner[j]
            itr = np.setdiff1d(np.arange(len(trainIdx)), val)
            m = SVC(C=C, kernel='linear').fit(X_train[itr], y_train[itr])
            accs.append(np.mean(m.predict(X_train[val]) == y_train[val]))
        if np.mean(accs) > bestAcc:
            bestAcc, bestC = np.mean(accs), C

    predHonest[testIdx] = SVC(C=bestC, kernel='linear').fit(X_train, y_train).predict(dat[testIdx])

print("\naccuracy when C is chosen only on training data: %.2f%%"
      % (np.mean(predHonest == realClassLabels) * 100))

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

**What is the cost parameter C?**

The SVM tries to find a line (a hyperplane) that separates the two classes. But our trials are not perfectly separable, some flexion trials look like extension trials. So the SVM is allowed to make mistakes, and **C decides how much those mistakes cost**.

- A **small C** means mistakes are cheap. The SVM does not try hard to get every training trial right and keeps the weights small. The border is smooth and simple. If C is too small, the classifier is too simple and misses the real pattern (underfitting).
- A **large C** means mistakes are expensive. The SVM bends the border to get as many training trials right as possible. If C is too large, it also learns the noise in the training data and works badly on new trials (overfitting).

So C controls the balance between "fit the training data" and "stay simple". You can see this in the output: with the small default C the weights have a norm of about 0.35, and when a much smaller C is picked the norm drops to about 0.09, so the model really does become simpler.

**Why is it optimized iteratively?**

Because there is no formula that gives the best C. It depends on the data, on how many features there are and how noisy they are. `ecogGetDefC` only gives a starting value (about 0.0032 here) using Joachims' method. To find something better, the code simply **tries many values** around it: it builds a list going up and down in steps of one third, trains an SVM for each one, and keeps the C that gave the highest accuracy. That is a grid search, and it is done inside every CV fold.

**Does it improve the predictions?**

At first it looks like it does:

| | accuracy |
| :--- | ---: |
| default C, no optimisation | 71.97% |
| with `optimizeC=True` | **77.39%** |

But this number is **too good, and it is not fair**. Looking at the function, the best C is chosen like this (lines 73-74):

```python
pred = svm.predict(X_test)
accuracy = np.mean(pred == y_test)
```

so C is selected by looking at the **test set**, and afterwards the accuracy is reported on that same test set. The classifier is allowed to peek at the answers before choosing its setting. That is why the accuracy goes up.

Two things show that something is wrong:

1. The best C is completely different in every fold, from 0.000055 up to 0.243. That is a factor of about 4000. If C really was a good setting for this data, it should be more or less the same everywhere. Jumping around like this means it is fitting the noise of each particular test set.
2. When I choose C **only from the training data** (I split the training data again into 3 inner parts and pick the C that works best there, which is called nested cross-validation), the accuracy is only **72.93%**.

So the honest result is: optimising C improves the accuracy from 71.97% to about **72.93%**, so roughly **1 percentage point**, and not the 5 points that `optimizeC=True` suggests. It also takes about 20 times longer, because 31 SVMs have to be trained in every fold instead of one.

My conclusion: optimising C is a reasonable idea, but it has to be done on the training data only. The way it is implemented here it mostly makes the result look better than it really is, and for this data the default C from `ecogGetDefC` is already a good choice.

<h2 style="color: #FF0000; font-weight: bold;">TASK 5 (1 pt):</h2>

Compare the results of the 3 different (LDA, Bayes and SVM) algorithms. Which one produces the best results (highest accuracy)?

Then also change your selected features (channels/frequencies), always keeping in mind the results from last times t-values/relief algorithm. 

Which features lead to the highest accuracy? (Also keep in mind that the more features you use the longer the calculation time is, so try to reduce your number of features without this resulting in a lower accuracy.)

In [ ]:
# --- TASK 5: compare the 3 classifiers and try different features ---
import warnings
warnings.filterwarnings('ignore')

def build_features(freqLo, freqHi, chans, step=1):
    """same preparation as in the cell above, but with freely chosen channels/frequencies"""
    fb = np.arange(freqLo, freqHi + 1, step)
    fIdx = np.unique(nearly(fb, ecog['periodogram']['centerFrequency']))
    ch = np.array(chans) - 1
    nF, nC = len(fIdx), len(ch)
    nTr = np.array(ecog['periodogram']['periodogram']).shape[2]

    d = np.array(ecog['periodogram']['periodogram'])[fIdx, :, :][:, ch, :]
    d = d.reshape(nF, nC * nTr, order='F')
    d = stats.zscore(d, axis=1)
    d = d.reshape(nF, nC, nTr, order='F')
    d = np.transpose(d, (2, 1, 0))
    d = d.reshape(nTr, nF * nC, order='F')
    return d[:len(realClassLabels), :]          # ecogStruct3 has 315 trials, epoch2 has 314

def run_cv(d, alg):
    """the same CV loop as above, just packed into a function"""
    pred = np.zeros(len(realClassLabels))
    for cv in range(1, N + 1):
        testIdx = np.where(selector == cv)[0]
        trainIdx = np.setdiff1d(np.arange(len(realClassLabels)), testIdx)
        labelIdx = getBalancedTrainset(realClassLabels[trainIdx].copy())
        tb = trainIdx[labelIdx]
        if alg == 'bayes':
            R = train_bayes(d[tb, :], realClassLabels[tb])
            pred[testIdx] = test_bayes(R, d[testIdx, :])['prediction']
        else:
            R = train_lda(d[tb, :], realClassLabels[tb])
            pred[testIdx] = test_lda(R, d[testIdx, :])['prediction']
    return np.mean(pred == realClassLabels)

feature_sets = [
    ('40-160 Hz, 5 channels (as given)', 40, 160, [17, 23, 29, 30, 39], 1),
    ('40-160 Hz, 8 channels',            40, 160, [16, 17, 22, 23, 29, 30, 39, 40], 1),
    ('40-160 Hz, 1 channel [17]',        40, 160, [17], 1),
    ('40-160 Hz every 10 Hz, 5 chan',    40, 160, [17, 23, 29, 30, 39], 10),
    ('40-160 Hz every 10 Hz, 4 chan',    40, 160, [17, 23, 30, 39], 10),
    ('40-160 Hz every 20 Hz, 2 chan',    40, 160, [17, 23], 20),
]

print('%-34s %6s %8s %8s %8s' % ('feature set', 'nFeat', 'bayes', 'lda', 'svm'))
for name, lo, hi, chs, st in feature_sets:
    d = build_features(lo, hi, chs, st)
    acc_b = run_cv(d, 'bayes')
    acc_l = run_cv(d, 'lda')
    acc_s = classification_svm(d, realClassLabels, selector, N, optimizeC=False)['accuracy']
    print('%-34s %6d %7.1f%% %7.1f%% %7.1f%%'
          % (name, d.shape[1], acc_b * 100, acc_l * 100, acc_s * 100))

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

**Which algorithm gives the best result?**

With the features that were given in the notebook (channels 17, 23, 29, 30, 39 and 40-160 Hz, which makes 310 features):

| classifier | accuracy |
| :--- | ---: |
| Bayes | 69.4% |
| LDA | 60.2% |
| **SVM** | **72.0%** |

So **SVM is the best**, Bayes is a bit behind, and LDA is clearly the worst. All of them are above 50%, which is what guessing would give with two balanced classes, so all three learned something.

The bad LDA result is not because LDA is a bad method. It is because we have **310 features but only 270 training trials**. LDA has to invert the within-class scatter matrix `s_w`, and with more features than trials that matrix cannot be inverted properly. This also showed up in Task 2, where `group1` and `group2` came out as huge numbers like 2e11. SVM does not have this problem, and Bayes does not either, because it treats every feature on its own.

**Which features give the highest accuracy?**

I tried different channels and different frequency steps:

| feature set | nFeat | Bayes | LDA | SVM |
| :--- | ---: | ---: | ---: | ---: |
| 40-160 Hz, 5 channels (as given) | 310 | 69.4% | 60.2% | 72.0% |
| 40-160 Hz, 8 channels | 496 | 71.0% | 54.5% | 72.0% |
| 40-160 Hz, 1 channel [17] | 62 | 68.5% | 65.9% | 72.9% |
| **40-160 Hz every 10 Hz, 4 channels** | **52** | 67.8% | 65.3% | **75.8%** |
| 40-160 Hz every 10 Hz, 5 channels | 65 | 69.1% | 65.3% | 75.2% |
| 40-160 Hz every 20 Hz, 2 channels | 14 | 67.5% | **68.8%** | 70.4% |

The best result is **SVM with only 52 features**: channels 17, 23, 30 and 39, and the frequencies from 40 to 160 Hz but only every 10 Hz. That gives **75.8%**, which is better than the 72.0% we get with all 310 features, and it uses **6 times fewer features** and runs much faster.

**What I learned from this:**

- **More features is not better.** Going up to 8 channels and 496 features did not help at all (SVM stayed at 72.0%), it only made everything slower. The neighbouring frequencies are very similar to each other, so they mostly repeat the same information instead of adding new information.
- **Taking every 10th frequency instead of every frequency helped.** This fits with the t-values from Session 5: the useful information is spread over a whole frequency band, not in single narrow lines, so a few frequencies per band are enough.
- **The channels matter more than the number of frequencies.** Channels 17 and 23 were the strongest in the t-value plot from last week, and even channel 17 alone already gives 72.9% with SVM.
- **LDA gets much better when there are fewer features**: from 60.2% with 310 features up to 68.8% with only 14. That is exactly what I expected from the `s_w` problem, and with few features LDA is even the best of the three.

So my final choice would be **SVM with the 52 features** (channels 17, 23, 30, 39 and 40-160 Hz in 10 Hz steps). It gives the highest accuracy, uses few features and is fast.

<h2 style="color: #FF0000; font-weight: bold;">TASK 6:</h2>

If you have time left, you can try to use the data from the PCA (load `resultsPCA.pkl` - data saved in `xPCA` from Session 04) to perform the classification.

Use the information you got last week from the t-values to choose the best features.